## Project: Labels vs. Scale: Breaking Even on Banking77 with Encoders and Decoders

**ModernBERT-base on Colab (G4 GPU)**: full fine-tuning (no LoRA) on every training budget and seed, under two epoch rules:

- **fixed3 (main):** every budget trains exactly 3 epochs, the same rule as the Qwen runs. Saved as model `modernbert`.
- **min300 (variant):** at least 300 optimizer steps per run (3 to 50 epochs). Shows whether small-budget results depend on training length. Saved as model `modernbert_min300`.

Each rule has its own learning-rate pilot at k10 (the best value depends on the number of steps). Run the cells from top to bottom: setup, smoke test, then the whole main protocol, then the variant. Everything uses the model after the final epoch.

Results go to Google Drive. If the session disconnects, reconnect, run **Setup** again, then run the **Run all** cell again: finished runs are skipped and training continues with the first unfinished one.

#### Setup

If you upload a newer version of a script to Drive, restart the session (Runtime, Restart session) and run Setup again, so Python loads the new version.

Mount Google Drive. Everything that must survive a disconnect (data backup, predictions, logs) lives there.

In [ ]:
# Mount Google Drive (a pop-up asks for permission the first time)
from google.colab import drive
drive.mount("/content/drive")

#### Paths and code

The code and data live on Drive and are copied to Colab's fast local disk at the start of every session. Only the results are written straight back to Drive.

In [ ]:
import os
import shutil

PROJECT = "/content/banking77"                      # Local copy of the repo (lost when the session ends)
DRIVE_DIR = "/content/drive/MyDrive/banking77"      # Permanent folder on Drive
RESULTS = f"{DRIVE_DIR}/results"                    # Predictions, training logs and pilot results are saved here

os.makedirs(RESULTS, exist_ok=True)
print("Results folder:", RESULTS)

In [ ]:
# Copy the code (scripts/) and the data (data/) from Drive to Colab's local disk. Nothing is downloaded from GitHub.
# Expected on Drive:  MyDrive/banking77/scripts  and  MyDrive/banking77/data (raw/ and splits/)
for folder in ["scripts", "data"]:
    source = f"{DRIVE_DIR}/{folder}"
    assert os.path.exists(source), f"Missing on Drive: {source}"
    shutil.copytree(source, f"{PROJECT}/{folder}", dirs_exist_ok=True)   # Overwrites older copies from an earlier session
print("Scripts:", sorted(os.listdir(f"{PROJECT}/scripts")))
print("Data folders:", sorted(os.listdir(f"{PROJECT}/data")))

In [ ]:
# Install what Colab does not already have (ModernBERT needs transformers 4.48 or newer)
!pip install -q "transformers>=4.48" pyarrow scikit-learn matplotlib

#### Check the setup

In [ ]:
import sys
import torch

os.chdir(PROJECT)                                   # The scripts expect to be run from the project root
sys.path.insert(0, f"{PROJECT}/scripts")

# GPU: the name should be the G4 (or A100) and bfloat16 should be supported
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None (change the runtime type)")
print("bfloat16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

# Data: the budgets and their sizes come from the split manifest
import evaluation as ev
print("Training budgets:", ev.budget_sizes())
print("Validation examples:", len(ev.split_labels("val")), "| Test examples:", len(ev.split_labels("test")))

#### Smoke test

One short run (k1, seed 0, main protocol) to check that everything works and to time it. It writes to a separate `smoke_test` folder on Drive, so it does not touch the real results. Divide its seconds by its steps (15) for a rough time per step, and multiply by the total steps of the real runs (about 11,000 for fixed3 and about 14,000 for min300).

In [ ]:
!python -u scripts/train_modernbert.py run --epoch-rule fixed3 --results-dir {DRIVE_DIR}/smoke_test --budgets k1 --seeds 0 --lr 5e-5

#### Main protocol: pilot (fixed3)

Trains k10 for 3 seeds and each learning rate in the grid, scored on the validation set only. The winner is saved to `results/pilot/best_lr_fixed3.json`. Safe to rerun after a disconnect: finished pilot runs are skipped.

In [ ]:
!python -u scripts/train_modernbert.py pilot --epoch-rule fixed3 --results-dir {RESULTS}

In [ ]:
# Pilot table: mean validation accuracy and loss for each learning rate (chosen = True is used for all fixed3 runs)
import diagnostics as dg
dg.summarize_pilot(RESULTS, rule="fixed3").round({"val_acc": 4, "val_loss": 4})

#### Main protocol: run all (fixed3)

Trains every budget and seed with the chosen learning rate. For each run it saves the val and test predictions, a training log and a config file to Drive. After a disconnect, run **Setup** and this cell again: it skips finished runs.

In [ ]:
!python -u scripts/train_modernbert.py run --epoch-rule fixed3 --results-dir {RESULTS}

#### Variant: pilot (min300)

The same pilot under the min300 rule. It has its own learning rate, saved to `results/pilot/best_lr_min300.json`.

In [ ]:
!python -u scripts/train_modernbert.py pilot --epoch-rule min300 --results-dir {RESULTS}

In [ ]:
# Pilot table for the min300 variant
dg.summarize_pilot(RESULTS, rule="min300").round({"val_acc": 4, "val_loss": 4})

#### Variant: run all (min300)

Trains every budget and seed under the min300 rule. Files are saved under the model name `modernbert_min300`, so they never overwrite the main protocol.

In [ ]:
!python -u scripts/train_modernbert.py run --epoch-rule min300 --results-dir {RESULTS}

#### Check the results

In [ ]:
# Progress: how many runs have both prediction files, for each protocol (24 expected for each)
import glob
for name in ["modernbert", "modernbert_min300"]:
    # The exact pattern keeps "modernbert" from also matching "modernbert_min300"
    n_done = len(glob.glob(f"{RESULTS}/{name}_k*_test.csv") + glob.glob(f"{RESULTS}/{name}_p*_test.csv") + glob.glob(f"{RESULTS}/{name}_full_*_test.csv"))
    print(name, "- finished runs:", n_done, "of", len(ev.budget_sizes()) * len(ev.SEEDS))

In [ ]:
# Test scores per budget (mean and spread over seeds) for both protocols
scores = ev.score_all("test", RESULTS)
scores = scores[scores["model"].isin(["modernbert", "modernbert_min300"])]
ev.summarize(scores).round(4)

In [ ]:
order = list(ev.budget_sizes())            # k1, k3, ..., p50, full
table = ev.summarize(scores)
table.assign(o=table["budget"].map(order.index)).sort_values("o").drop(columns="o").round(4)

In [ ]:
# Loss and validation-accuracy curves for one budget; change the model name and budget to look at another
dg.plot_training_curves("modernbert", "k10", results_dir=RESULTS)

In [ ]:
dg.plot_training_curves("modernbert_min300", "k10", results_dir=RESULTS)